# PhishGuard Model Training Pipeline

This notebook outlines the training workflow for the phishing URL detection model.

## 1. Title & Introduction
- TODO: Provide project overview and training objectives.

## 2. Imports
- TODO: List required libraries for data processing, feature extraction, and modeling.

## 3. Load Dataset
- TODO: Document dataset paths from `data/raw/` and `data/processed/`.
- TODO: Include notes about combining multiple phishing sources.

## 4. Data Cleaning / URL Normalization
- TODO: Outline URL normalization strategy (lowercasing, punycode handling, etc.).
- TODO: Plan for deduplication and suspicious pattern heuristics.

## 5. Feature Extraction
- Reference implementation in `src/features/url_features.py`.
- TODO: Document expected feature schema and transformations.

## 6. Temporal Train/Validation/Test Split
- TODO: Describe chronological split strategy to respect URL discovery times.
- TODO: Define validation window size and holdout period.

## 7. Model Selection Plan
- Baseline: Logistic Regression.
- Candidate: LightGBM.
- TODO: Establish criteria (accuracy, latency, explainability) for model choice.

## 8. Training Pipeline
- TODO: Design pipeline structure (data loaders, preprocessors, model training loop).
- TODO: Outline hyperparameter tuning approach.

## 9. Evaluation Plan
- TODO: Define metrics — precision, recall, F1-score, ROC-AUC.
- TODO: Plan for confusion matrix and calibration analysis.

## 10. Model Saving Step
- TODO: Outline joblib persistence workflow and model versioning approach.

## 11. Next Steps
- TODO: Detail integration milestones for FastAPI service endpoints.
- TODO: Plan Chrome extension communication workflow and UI updates.

---
_Notebook prepared as a scaffold; fill in each section with detailed steps and code as development progresses._

In [ ]:
# Baseline environment setup and imports
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Ensure src package is importable when running notebook from notebooks/
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.features import extract_all_features  # noqa: E402

In [ ]:
# Construct temporary in-memory dataset
url_samples = [
    ("https://google.com", 0),
    ("http://192.168.0.14/login", 1),
    ("https://secure-paypal.com.verify-login.tj", 1),
    ("https://microsoft.com", 0),
]
baseline_df = pd.DataFrame(url_samples, columns=["url", "label"])
baseline_df

In [ ]:
# Generate baseline feature matrix
feature_rows = [extract_all_features(url) for url in baseline_df["url"]]
features_df = pd.DataFrame(feature_rows)
X_baseline = features_df
y_baseline = baseline_df["label"]
features_df

In [ ]:
# Train baseline Logistic Regression on the entire dataset (sanity check)
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_baseline, y_baseline)

y_pred = baseline_model.predict(X_baseline)
accuracy = accuracy_score(y_baseline, y_pred)
precision = precision_score(y_baseline, y_pred, zero_division=0)
recall = recall_score(y_baseline, y_pred, zero_division=0)
f1 = f1_score(y_baseline, y_pred, zero_division=0)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 Score : {f1:.3f}")

In [ ]:
# Persist baseline model for reference
models_dir = repo_root / "models"
models_dir.mkdir(exist_ok=True)
model_path = models_dir / "baseline_model.pkl"
joblib.dump(baseline_model, model_path)
model_path